Cell 1: setup

In [ ]:
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
from shapely.geometry import Polygon

from bearcast.models.evaluate import capture_by_month

D = Path("../data/processed")
preds = pd.read_parquet(D / "preds_h3r7_test.parquet")
preds["cell"] = preds["cell"].astype(str)
fc_path = sorted(D.glob("forecast_h3r7_*.parquet"))[-1]
fc = pd.read_parquet(fc_path)

cells = pd.read_parquet(D / "cells_h3r7.parquet", columns=["cell"])
hexes = gpd.GeoDataFrame(cells, geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells["cell"]],
                         crs=4326).to_crs(3087)
print(f"{len(hexes):,} hexagons | test months {preds.month.min():%Y-%m} to {preds.month.max():%Y-%m} | forecast file {fc_path.name}")

def basemap(ax, title):
    hexes.plot(ax=ax, color="#eeeeee", linewidth=0)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()

Cell 2: busiest test month, predicted vs actual

In [ ]:
month = preds.groupby("month")["n_conflict"].sum().idxmax()          # busiest test month
m = hexes.merge(preds[preds["month"] == month], on="cell")
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
basemap(axes[0], f"Predicted conflict calls, {month:%B %Y}")
m.plot(ax=axes[0], column="gbm_full", cmap="YlOrRd", norm=LogNorm(vmin=1e-3, vmax=m["gbm_full"].max()),
       linewidth=0, legend=True, legend_kwds={"shrink": 0.6, "label": "expected calls (log scale)"})
basemap(axes[1], f"Actual conflict calls, {month:%B %Y} ({int(m.n_conflict.sum())} calls)")
top = m["gbm_full"].rank(ascending=False) <= 0.05 * len(m)
m[top].plot(ax=axes[1], color="#fdd49e", linewidth=0)
act = m[m["n_conflict"] > 0]
act.assign(geometry=act.geometry.centroid).plot(ax=axes[1], color="#b30000", markersize=act["n_conflict"] * 6)
axes[1].text(0.02, 0.02, "shaded: model's top 5% of hexagons\ndots: actual calls (size = count)",
             transform=axes[1].transAxes, fontsize=9)
caught = act.loc[top[act.index], "n_conflict"].sum() / act["n_conflict"].sum()
print(f"{month:%B %Y}: {caught:.0%} of conflict calls fell in the model's top 5% of hexagons")
plt.tight_layout()

Cell 3: emerging risk in that month

In [ ]:
cold = preds[preds["conf_sum12"].eq(0)].copy()
cold["pct"] = cold.groupby("month")["gbm_full"].rank(ascending=False, pct=True)
c = hexes.merge(cold[cold["month"] == month], on="cell")
fig, ax = plt.subplots(figsize=(8, 8))
basemap(ax, f"Emerging risk, {month:%B %Y}: hexagons with no conflict in the prior 12 months")
c[c["pct"] <= 0.10].plot(ax=ax, color="#9ecae1", linewidth=0)
new = c[c["n_conflict"] > 0]
new.assign(geometry=new.geometry.centroid).plot(ax=ax, color="#08306b", markersize=12)
hit = new["pct"].le(0.10).mean() if len(new) else float("nan")
ax.text(0.02, 0.02, f"blue: model's top 10% of these hexagons\ndots: conflicts that happened there ({hit:.0%} inside the blue)",
        transform=ax.transAxes, fontsize=9)
plt.tight_layout()

Cell 4: capture curve over the whole test period

In [ ]:
ks = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
curve = pd.DataFrame({s: [capture_by_month(preds, s, k).mean() for k in ks]
                      for s in ["base_longrun", "gbm_history", "gbm_full"]}, index=[f"{k:.1%}" for k in ks])
ax = curve.plot(marker="o", figsize=(7, 4.5))
ax.set_xlabel("share of hexagons watched (top k%)"); ax.set_ylabel("share of conflict calls caught")
ax.set_title("Test period 2025-01 to 2026-10: how much of the trouble each ranking catches")
ax.grid(alpha=0.3)
print(curve.round(3).to_string())

Cell 5: next month’s forecast

In [ ]:
f = hexes.merge(fc, on="cell")
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
basemap(axes[0], f"Forecast {fc['month'].iloc[0]:%B %Y}: risk tiers")
colors = {"very high": "#b30000", "high": "#fc8d59", "elevated": "#fdd49e"}
for tier, col in colors.items():
    f[f["tier"] == tier].plot(ax=axes[0], color=col, linewidth=0, label=tier)
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], labels=list(colors), loc="lower left")
basemap(axes[1], f"Forecast {fc['month'].iloc[0]:%B %Y}: emerging risk (top 5% of quiet hexagons)")
f[f["emerging_pct"].le(0.05).fillna(False)].plot(ax=axes[1], color="#2171b5", linewidth=0)
print(f"expected calls statewide: {fc['expected_calls'].sum():.0f}")
plt.tight_layout()